# 08 — A complete forecasting workflow

This capstone notebook chains the whole pipeline on Sioux Falls, mirroring the
AequilibraE *Forecasting* documentation example:

1. **base-year assignment** with skimming;
2. **gravity model calibration** on congested times;
3. **future demand**: grown trip ends balanced with IPF;
4. **future-year assignment** with **select link analysis**;
5. compare base vs future flows on a map.


In [1]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import numpy as np
import pandas as pd

from aequilibrae.utils.create_example import create_example
from aequilibrae.paths import TrafficAssignment, TrafficClass

np.random.seed(0)

fldr = str(Path(gettempdir()) / uuid4().hex)
project = create_example(fldr, "sioux_falls")

project.network.build_graphs()
graph = project.network.graphs["c"]
graph.set_graph("free_flow_time")
graph.set_skimming(["free_flow_time", "distance"])
graph.set_blocked_centroid_flows(False)

C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  build_compressed_graph(self, remove_dead_ends)
C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are settin

In [2]:
# ---- 1. Base year assignment --------------------------------------------
demand = project.matrices.get_matrix("demand_omx")
demand.computational_view(["matrix"])

assigclass = TrafficClass(name="car", graph=graph, matrix=demand)
assig = TrafficAssignment()
assig.add_class(assigclass)
assig.set_vdf("BPR")
assig.set_vdf_parameters({"alpha": "b", "beta": "power"})
assig.set_capacity_field("capacity")
assig.set_time_field("free_flow_time")
assig.set_algorithm("bfw")
assig.max_iter = 500
assig.rgap_target = 0.001
assig.execute()

assig.save_results("base_year_assignment")
assig.save_skims("base_year_skims", which_ones="all", format="omx")
base_flows = assig.results()[["matrix_tot"]].rename(columns={"matrix_tot": "base"})

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

In [3]:
# ---- 2. Gravity calibration on congested skims --------------------------
from aequilibrae.distribution import GravityCalibration

imped = project.matrices.get_matrix("base_year_skims_car")
imped.computational_view(["free_flow_time_final"])   # congested time, last iteration

np.fill_diagonal(imped.matrix_view, 0)
intrazonal = 0.75 * np.amin(imped.matrix_view, where=imped.matrix_view > 0,
                            initial=imped.matrix_view.max(), axis=1)
np.fill_diagonal(imped.matrix_view, intrazonal)
imped.save(names=["time_with_intrazonals"])

gc = GravityCalibration(matrix=demand, impedance=imped, function="power", nan_as_zero=True)
gc.calibrate()
gc.model.save(str(Path(fldr) / "power_model.mod"))

In [4]:
# ---- 3. Future demand: grown vectors + IPF -------------------------------
from aequilibrae.distribution import Ipf

origins = np.sum(demand.matrix_view, axis=1)
destinations = np.sum(demand.matrix_view, axis=0)
orig = origins * (1 + np.random.rand(origins.shape[0]) / 10)
dest = destinations * (1 + np.random.rand(origins.shape[0]) / 10)
dest *= orig.sum() / dest.sum()

vectors = pd.DataFrame({"origins": orig, "destinations": dest}, index=demand.index[:])

ipf = Ipf(matrix=demand, vectors=vectors, row_field="origins",
          column_field="destinations", nan_as_zero=True)
ipf.fit()
ipf.save_to_project(name="demand_future", file_name="demand_future.omx")

In [5]:
# ---- 4. Future-year assignment with select link analysis -----------------
future = project.matrices.get_matrix("demand_future")
future.computational_view("matrix")

assigclass = TrafficClass(name="car", graph=graph, matrix=future)
assigclass.set_select_links({
    "downtown_bridge": [(13, 1), (25, 1)],   # (link_id, direction) tuples
})

assig = TrafficAssignment()
assig.add_class(assigclass)
assig.set_vdf("BPR")
assig.set_vdf_parameters({"alpha": "b", "beta": "power"})
assig.set_capacity_field("capacity")
assig.set_time_field("free_flow_time")
assig.set_algorithm("bfw")
assig.max_iter = 500
assig.rgap_target = 0.001
assig.execute()

assig.save_results("future_year_assignment")
assig.save_select_link_results("select_link_analysis")

future_flows = assig.results()[["matrix_tot"]].rename(columns={"matrix_tot": "future"})

[interactive offline map - run the notebook to display]

[interactive offline map - run the notebook to display]

In [6]:
# ---- 5. Compare ----------------------------------------------------------
comparison = base_flows.join(future_flows)
comparison["growth_pct"] = (100 * (comparison.future / comparison.base - 1)).round(1)
comparison.sort_values("growth_pct", ascending=False).head(8)

,base,future,growth_pct
link_id,,,
1,4567.230095,5452.502999,19.4
3,4574.375228,5459.857040,19.4
55,15713.659996,18000.222079,14.6
2,8174.162682,9348.015078,14.4
60,19162.405854,21596.144990,12.7
7,10207.585261,11349.011180,11.2
35,10172.571513,11286.688037,11.0
6,14184.787941,15708.398419,10.7


In [7]:
# Offline map helper ---------------------------------------------------------
# Interactive maps with no server extensions, no labextensions beyond the
# ipywidgets manager, and no CDN: lonboard renders WebGL maps whose frontend
# JavaScript ships from the kernel through the ipywidgets channel.
#
# Backends (AEQ_MAP_BACKEND environment variable):
#   lonboard (default) - interactive WebGL maps (pip install lonboard anywidget)
#   static             - matplotlib rendering, works absolutely anywhere
#
# The declarative symbology below (field()/constant() chains) is self-contained
# and renders identically on both backends.
import os

import matplotlib.colors
import matplotlib.pyplot as _plt
import numpy as np


# --- declarative symbology --------------------------------------------------
class _Mapping:
    def __init__(self, field, scheme, params):
        self.field, self.scheme, self.params = field, scheme, params

    def encoding(self, *targets):
        return {"field": self.field, "scheme": self.scheme,
                "params": self.params, "encodings": list(targets)}


class _Field:
    def __init__(self, name):
        self.name = name

    def colormap(self, name="viridis", *, domain=None, reverse=False, n_shades=9):
        return _Mapping(self.name, "colormap",
                        {"name": name, "domain": domain, "reverse": reverse})

    def scalar(self, *, domain, output_range):
        return _Mapping(self.name, "scalar",
                        {"domain": list(domain), "range": list(output_range)})

    def categorical(self, name="tab10"):
        return _Mapping(self.name, "categorical", {"name": name})


class _Constant:
    def __init__(self, value):
        self.value = value

    def encoding(self, *targets):
        scheme = "constant_num" if isinstance(self.value, (int, float)) else "constant_color"
        return {"field": None, "scheme": scheme,
                "params": {"value": self.value}, "encodings": list(targets)}


def field(name):
    """Style by a data column: .colormap() / .scalar() / .categorical()."""
    return _Field(name)


def constant(value):
    """A fixed colour (hex/name) or number, e.g. constant("#dc2626")."""
    return _Constant(value)


def _rgba255(c, alpha=1.0):
    r, g, b, a = matplotlib.colors.to_rgba(c, alpha)
    return [int(r * 255), int(g * 255), int(b * 255), int(a * 255)]


def _style_arrays(symbology, gdf):
    """symbology -> per-row uint8 RGBA arrays and float width arrays."""
    n = len(gdf)
    out = {"stroke": None, "width": None, "fill": None}
    if not symbology:
        return out
    mappings = [m for group in symbology for m in (group if isinstance(group, list) else [group])]
    for m in mappings:
        scheme, params, fld, encs = m["scheme"], m["params"], m["field"], m["encodings"]
        arr = wid = None
        if scheme == "constant_color":
            arr = np.tile(_rgba255(params["value"]), (n, 1)).astype(np.uint8)
        elif scheme == "colormap":
            cmap = _plt.get_cmap(params["name"])
            if params.get("reverse"):
                cmap = cmap.reversed()
            dom = params.get("domain") or [float(gdf[fld].min()), float(gdf[fld].max())]
            vals = gdf[fld].to_numpy(dtype=float)
            t = np.clip((vals - dom[0]) / max(dom[1] - dom[0], 1e-12), 0, 1)
            rgba = cmap(t)
            arr = (rgba * 255).astype(np.uint8)
        elif scheme == "categorical":
            cmap = _plt.get_cmap(params["name"])
            uniq = list(dict.fromkeys(gdf[fld].dropna()))
            idx = {v: i for i, v in enumerate(uniq)}
            arr = np.array([_rgba255(cmap(idx.get(v, 0) % cmap.N)) for v in gdf[fld]], dtype=np.uint8)
        elif scheme == "constant_num":
            wid = np.full(n, float(params["value"]))
        elif scheme == "scalar":
            d, r = params["domain"], params["range"]
            vals = gdf[fld].to_numpy(dtype=float)
            t = np.clip((vals - d[0]) / max(d[1] - d[0], 1e-12), 0, 1)
            wid = r[0] + t * (r[1] - r[0])
        if arr is not None:
            if any("stroke" in e for e in encs):
                out["stroke"] = arr
            if any("fill" in e for e in encs):
                out["fill"] = arr
        if wid is not None and any("width" in e for e in encs):
            out["width"] = wid
    return out


# --- the map document -------------------------------------------------------
class MapDoc:
    """Collects styled layers; displays via lonboard (WebGL) or matplotlib."""

    def __init__(self):
        self.items = []  # (gdf, name, arrays, opacity)

    def add(self, gdf, name, symbology, opacity):
        g = gdf.reset_index(drop=True).explode(index_parts=False).reset_index(drop=True)
        self.items.append((g, name, _style_arrays(symbology, g), opacity))

    def _lonboard_map(self):
        from lonboard import Map, PathLayer, PolygonLayer, ScatterplotLayer
        layers = []
        for g, name, st, op in self.items:
            if not len(g):
                continue
            geom = g.geometry.geom_type.iloc[0]
            base = g[["geometry"]]
            if "LineString" in geom:
                kw = {"width_units": "pixels", "width_min_pixels": 1.0, "opacity": op}
                if st["stroke"] is not None:
                    kw["get_color"] = st["stroke"]
                if st["width"] is not None:
                    kw["get_width"] = st["width"]
                layers.append(PathLayer.from_geopandas(base, **kw))
            elif "Polygon" in geom:
                kw = {"opacity": op * 0.6, "stroked": False}
                if st["fill"] is not None:
                    kw["get_fill_color"] = st["fill"]
                layers.append(PolygonLayer.from_geopandas(base, **kw))
            else:
                kw = {"radius_min_pixels": 5, "opacity": op}
                fill = st["fill"] if st["fill"] is not None else st["stroke"]
                if fill is not None:
                    kw["get_fill_color"] = fill
                layers.append(ScatterplotLayer.from_geopandas(base, **kw))
        return Map(layers=layers, basemap=None)

    def _static_figure(self):
        fig, ax = _plt.subplots(figsize=(9, 7))
        ax.set_facecolor("#eef1f4")
        for g, name, st, op in self.items:
            if not len(g):
                continue
            geom = g.geometry.geom_type.iloc[0]
            if "LineString" in geom:
                colors = st["stroke"] / 255 if st["stroke"] is not None else "#1d4ed8"
                widths = st["width"] if st["width"] is not None else 1.0
                g.plot(ax=ax, color=colors, linewidth=widths, alpha=op)
            elif "Polygon" in geom:
                colors = st["fill"] / 255 if st["fill"] is not None else "#cbd5e1"
                g.plot(ax=ax, color=colors, alpha=op * 0.6)
            else:
                fill = st["fill"] if st["fill"] is not None else st["stroke"]
                g.plot(ax=ax, color=(fill / 255 if fill is not None else "#dc2626"),
                       markersize=25, alpha=op)
        ax.set_aspect(1.4)
        ax.set_xticks([]), ax.set_yticks([])
        _plt.tight_layout()
        _plt.close(fig)
        return fig

    def _ipython_display_(self):
        from IPython.display import display
        be = os.environ.get("AEQ_MAP_BACKEND", "lonboard").strip().lower()
        display(self._static_figure() if be == "static" else self._lonboard_map())


def new_map(gdf_for_extent=None, zoom=12):
    """Create a map document (extent/zoom args kept for API compatibility;
    lonboard auto-fits to its layers)."""
    return MapDoc()


def add_gdf(doc, gdf, name, symbology=None, **kwargs):
    """Add a GeoDataFrame to the map as a styled layer."""
    doc.add(gdf, name, symbology, kwargs.get("opacity", 1.0))
    return name


def merge_lines(gdf, tol=0.01):
    """Collapse many lines into a single MultiLineString feature — backdrop
    layers do not need per-feature identity, and one merged feature is a
    fraction of the size and draw cost."""
    import geopandas as _gpd
    from shapely.geometry import MultiLineString
    parts = []
    for geom in gdf.geometry.simplify(tol):
        if geom is None or geom.is_empty:
            continue
        parts.extend(geom.geoms if geom.geom_type == "MultiLineString" else [geom])
    return _gpd.GeoDataFrame({"links": [len(parts)]}, geometry=[MultiLineString(parts)], crs=gdf.crs)


In [8]:
# field()/constant() symbology builders come from the map helper cell

links = project.network.links.data
gdf = links.merge(comparison.reset_index(), on="link_id")
gdf["growth_pct"] = gdf["growth_pct"].fillna(0)

doc = new_map(gdf, zoom=12)
lim = float(np.nanmax(np.abs(gdf["growth_pct"])))
add_gdf(doc, gdf[["link_id", "growth_pct", "base", "future", "geometry"]], "flow growth %",
        symbology=[[field("growth_pct").colormap("RdBu_r", domain=(-lim, lim)).encoding("stroke"),
                    field("future").scalar(domain=(0.0, float(gdf["future"].max())),
                                           output_range=(0.8, 7.0)).encoding("stroke-width")]])
doc

[interactive offline map - run the notebook to display]

In [9]:
project.close()

---
That completes the series. Recap of the toolkit:

| Notebook | Modeling stage | Key classes |
|---|---|---|
| 01 | Project & network | `Project`, `Network` |
| 02 | Zoning & connectors | `Zoning`, `Zone.connect_mode` |
| 03 | Paths & skims | `Graph`, `PathResults`, `NetworkSkimming` |
| 04 | Distribution | `GravityCalibration`, `Ipf`, `GravityApplication` |
| 05 | Assignment | `TrafficAssignment`, `TrafficClass` |
| 06 | Route choice | `RouteChoice` |
| 07 | Transit | `Transit`, GTFS builder |
| 08 | Forecasting | everything together |

Further reading: the [AequilibraE documentation](https://www.aequilibrae.com) —
every notebook here mirrors one or more of its worked examples.
